In [1]:
import pandas as pd
from io import StringIO
from datetime import datetime

# --- FUNKTIONER FÖR ATT LÄSA OLIKA FILTYPER ---

def load_el_data(filenames):
    """Läser el-filer med ; och decimal-komma"""
    dfs = []
    for f in filenames:
        print(f"Läser el-fil: {f}")
        # Läser med rätt avgränsare och decimaltecken
        df = pd.read_csv(f, sep=';', decimal=',', encoding='utf-8')
        df['timestamp'] = pd.to_datetime(df['Datum'])

        # Standardisera kolumnnamn
        df = df.rename(columns={'El kWh': 'consumption_kwh'})

        # Hantera kolumnnamn för pris (standardiserar till price_total_ore)
        if 'Totalt (öre/kWh)' in df.columns:
            df = df.rename(columns={'Totalt (öre/kWh)': 'price_total_ore'})
        else:
            print(f"Varning: Hittade inte pris-kolumn i {f}")
            
        dfs.append(df[['timestamp', 'consumption_kwh', 'price_total_ore']])

    out = pd.concat(dfs).drop_duplicates('timestamp')
    return out

def load_solis_data(filenames):
    """Läser inverter-filer med , och decimal-punkt"""
    dfs = []
    for f in filenames:
        print(f"Läser Solis-fil: {f}")
        df = pd.read_csv(f)
        df['timestamp'] = pd.to_datetime(df['timestamp'])
        df = df.rename(columns={'avg_power_kw': 'production_kw'})
        dfs.append(df[['timestamp', 'production_kw']])

    out = pd.concat(dfs).drop_duplicates('timestamp')
    return out

def load_nasa_data(filename):
    """Läser NASA-filen och konverterar LST (UTC+1) till lokal svensk tid."""
    print(f"Läser NASA-fil: {filename}")
    with open(filename, 'r') as f:
        lines = f.readlines()

    # Hitta exakt rad där datat börjar
    start_idx = next(i for i, line in enumerate(lines) if line.startswith('YEAR'))
    data_str = ''.join(lines[start_idx:])
    df = pd.read_csv(StringIO(data_str))

    # Skapa timestamp (LST = UTC+1 = Etc/GMT-1)
    df['timestamp'] = pd.to_datetime(
        df[['YEAR', 'MO', 'DY', 'HR']].rename(
            columns={'YEAR': 'year', 'MO': 'month', 'DY': 'day', 'HR': 'hour'}
        )
    )
    # NASA LST är alltid UTC+1
    df['timestamp'] = df['timestamp'].dt.tz_localize('Etc/GMT-1')

    # Konvertera till Europe/Stockholm (hanterar sommar/vintertid automatiskt)
    df['timestamp'] = df['timestamp'].dt.tz_convert('Europe/Stockholm').dt.tz_localize(None)

    df = df.rename(columns={'ALLSKY_SFC_SW_DWN': 'irradiance_wm2'})
    return df[['timestamp', 'irradiance_wm2']]

def print_time_info(name, df):
    print(f"\n--- {name} ---")
    print(f"Antal rader: {len(df)}")
    print(f"Intervall: {df['timestamp'].min()} till {df['timestamp'].max()}")

# --- GENOMFÖR SAMMANSLAGNINGEN ---

print("Laddar och tvättar data...\n")

# 1. El-data (Uppdaterade filnamn)
el_files = [
    'El - Erlandsliden 15 B, Halmstad 2023 - halvår.csv',
    'El - Erlandsliden 15 B, Halmstad 2024.csv',
    'El - Erlandsliden 15 B, Halmstad 2025.csv'
]
df_el = load_el_data(el_files)
print_time_info("EL-data", df_el)

# 2. Inverter-data (Uppdaterade filnamn)
solis_files = [
    'solis_data_2023_korrigerad.csv',
    'solis_data_2024_local.csv',
    'solis_data_2025_local.csv'
]
df_solis = load_solis_data(solis_files)
print_time_info("Solis-data", df_solis)

# 3. NASA-data (Uppdaterat filnamn)
df_nasa = load_nasa_data('POWER_Point_Hourly_20230701_20260323_056d66N_012d81E_LST.csv')
print_time_info("NASA-data", df_nasa)

# --- MERGE TILL MASTER ---
print("\nSlår samman till Master DataFrame...")
master_df = pd.merge(df_el, df_solis, on='timestamp', how='left')
master_df = pd.merge(master_df, df_nasa, on='timestamp', how='left')

# Fyll produktion med 0
master_df['production_kw'] = master_df['production_kw'].fillna(0)
master_df = master_df.sort_values('timestamp').reset_index(drop=True)

# Spara Master-filen
master_df.to_csv('master_energy_data.csv', index=False)

print("\n--- KLART! master_energy_data.csv har skapats ---")
display(master_df.head())

# Korrelationskoll (Bör vara hög om tiderna är synkade)
corr = master_df['production_kw'].corr(master_df['irradiance_wm2'])
print(f"\nSynk-kontroll: Korrelation Sol/Produktion är {corr:.2f}")

Laddar och tvättar data...

Läser el-fil: El - Erlandsliden 15 B, Halmstad 2023 - halvår.csv
Läser el-fil: El - Erlandsliden 15 B, Halmstad 2024.csv
Läser el-fil: El - Erlandsliden 15 B, Halmstad 2025.csv

--- EL-data ---
Antal rader: 21958
Intervall: 2023-07-01 00:00:00 till 2025-12-31 23:00:00
Läser Solis-fil: solis_data_2023_korrigerad.csv
Läser Solis-fil: solis_data_2024_local.csv
Läser Solis-fil: solis_data_2025_local.csv

--- Solis-data ---
Antal rader: 21917
Intervall: 2023-07-01 04:00:00 till 2025-12-31 16:00:00
Läser NASA-fil: POWER_Point_Hourly_20230701_20260323_056d66N_012d81E_LST.csv

--- NASA-data ---
Antal rader: 21960
Intervall: 2023-07-01 01:00:00 till 2025-12-31 23:00:00

Slår samman till Master DataFrame...

--- KLART! master_energy_data.csv har skapats ---


,timestamp,consumption_kwh,price_total_ore,production_kw,irradiance_wm2
0,2023-07-01 00:00:00,0.8,0.000,0.000000,NaN
1,2023-07-01 01:00:00,2.4,34.859,0.000000,0.00
2,2023-07-01 02:00:00,6.0,34.481,0.000000,0.00
3,2023-07-01 03:00:00,2.6,34.232,0.000000,0.00
4,2023-07-01 04:00:00,0.6,34.008,23.333333,1.98



Synk-kontroll: Korrelation Sol/Produktion är 0.89


In [2]:
import pandas as pd
from io import StringIO
import numpy as np

# --- 1. FUNKTIONER FÖR INLÄSNING ---

def load_el_data(filenames):
    """Läser el-filer från leverantör (; avgränsare, , decimal)"""
    dfs = []
    for f in filenames:
        print(f"Bearbetar el-leverantörsdata: {f}")
        df = pd.read_csv(f, sep=';', decimal=',', encoding='utf-8')
        df['timestamp'] = pd.to_datetime(df['Datum'])
        
        # Mappa till tydliga namn för flödesanalys
        df = df.rename(columns={
            'El kWh': 'grid_import_kwh',
            'Produktion': 'grid_export_kwh'
        })
        
        # Säkerställ priskolumn
        if 'Totalt (öre/kWh)' in df.columns:
            df = df.rename(columns={'Totalt (öre/kWh)': 'price_total_ore'})
        
        dfs.append(df[['timestamp', 'grid_import_kwh', 'grid_export_kwh', 'price_total_ore']])
    return pd.concat(dfs).drop_duplicates('timestamp')

def load_solis_data(filenames):
    """Läser inverter-data och konverterar W till kW"""
    dfs = []
    for f in filenames:
        print(f"Bearbetar Solis-data: {f}")
        df = pd.read_csv(f)
        df['timestamp'] = pd.to_datetime(df['timestamp'])
        
        # Konvertera från Watt till kW (eftersom mätvärdena i CSV är t.ex. 5000 för 5kW)
        df['total_production_kw'] = pd.to_numeric(df['avg_power_kw'], errors='coerce') / 1000.0
        dfs.append(df[['timestamp', 'total_production_kw']])
    return pd.concat(dfs).drop_duplicates('timestamp')

def load_nasa_data(filename):
    """Läser NASA-data och justerar LST (UTC+1) till Svensk Lokaltid (UTC+1/+2)"""
    print(f"Bearbetar NASA-väderdata: {filename}")
    with open(filename, 'r') as f:
        lines = f.readlines()
    start_idx = next(i for i, line in enumerate(lines) if line.startswith('YEAR'))
    df = pd.read_csv(StringIO(''.join(lines[start_idx:])))
    
    # Skapa timestamp
    df['timestamp'] = pd.to_datetime(df[['YEAR', 'MO', 'DY', 'HR']].rename(
        columns={'YEAR': 'year', 'MO': 'month', 'DY': 'day', 'HR': 'hour'}))
    
    # Justera tid: NASA LST är fast UTC+1. Konvertera till Europe/Stockholm för att få sommar/vintertid.
    df['timestamp'] = df['timestamp'].dt.tz_localize('Etc/GMT-1').dt.tz_convert('Europe/Stockholm').dt.tz_localize(None)
    
    df = df.rename(columns={'ALLSKY_SFC_SW_DWN': 'irradiance_wm2'})
    return df[['timestamp', 'irradiance_wm2']]

# --- 2. KÖR BEARBETNING ---

# Definiera dina filnamn exakt som de heter i mappen
el_files = [
    'El - Erlandsliden 15 B, Halmstad 2023 - halvår.csv',
    'El - Erlandsliden 15 B, Halmstad 2024.csv',
    'El - Erlandsliden 15 B, Halmstad 2025.csv'
]
solis_files = [
    'solis_data_2023_korrigerad.csv',
    'solis_data_2024_local.csv',
    'solis_data_2025_local.csv'
]
nasa_file = 'POWER_Point_Hourly_20230701_20260323_056d66N_012d81E_LST.csv'

# Ladda all data
df_grid = load_el_data(el_files)
df_inv = load_solis_data(solis_files)
df_weather = load_nasa_data(nasa_file)

# Slå samman till Master DataFrame
master_df = pd.merge(df_grid, df_inv, on='timestamp', how='left')
master_df = pd.merge(master_df, df_weather, on='timestamp', how='left')

# --- 3. BERÄKNA ENERGIFLÖDEN ("Räkna baklänges") ---

# Fyll produktion som saknas (nätter) med 0
master_df['total_production_kw'] = master_df['total_production_kw'].fillna(0)

# A. Egenförbrukning = Det som producerats minus det som skickats ut på nätet
# (Vi använder .clip(0) för att undvika små negativa värden pga mätardifferenser)
master_df['self_consumption_kwh'] = (master_df['total_production_kw'] - master_df['grid_export_kwh']).clip(lower=0)

# B. Fastighetens totala last (verkligt behov) = Inköpt el + Egenförbrukning
master_df['total_house_load_kwh'] = master_df['grid_import_kwh'] + master_df['self_consumption_kwh']

# Sortera och spara
master_df = master_df.sort_values('timestamp').reset_index(drop=True)
master_df.to_csv('master_energy_data_v2.csv', index=False)

print("\n" + "="*30)
print("KLART! Din master-fil 'master_energy_data_v2.csv' är skapad.")
print(f"Antal rader: {len(master_df)}")
print("="*30)
print(master_df.head())

Bearbetar el-leverantörsdata: El - Erlandsliden 15 B, Halmstad 2023 - halvår.csv
Bearbetar el-leverantörsdata: El - Erlandsliden 15 B, Halmstad 2024.csv
Bearbetar el-leverantörsdata: El - Erlandsliden 15 B, Halmstad 2025.csv
Bearbetar Solis-data: solis_data_2023_korrigerad.csv
Bearbetar Solis-data: solis_data_2024_local.csv
Bearbetar Solis-data: solis_data_2025_local.csv
Bearbetar NASA-väderdata: POWER_Point_Hourly_20230701_20260323_056d66N_012d81E_LST.csv

KLART! Din master-fil 'master_energy_data_v2.csv' är skapad.
Antal rader: 21961
            timestamp  grid_import_kwh  grid_export_kwh  price_total_ore  \
0 2023-07-01 00:00:00              0.8              0.0            0.000   
1 2023-07-01 01:00:00              2.4              0.0           34.859   
2 2023-07-01 02:00:00              6.0              0.0           34.481   
3 2023-07-01 03:00:00              2.6              0.0           34.232   
4 2023-07-01 04:00:00              0.6              0.0           34.008   

 

In [3]:
import pandas as pd
from io import StringIO
from datetime import datetime
import numpy as np

# --- 1. FUNKTIONER FÖR INLÄSNING ---

def load_el_data(filenames):
    """Läser el-filer från leverantör (; avgränsare, , decimal)."""
    dfs = []
    for f in filenames:
        print(f"Bearbetar el-leverantörsdata: {f}")
        df = pd.read_csv(f, sep=';', decimal=',', encoding='utf-8')
        df['timestamp'] = pd.to_datetime(df['Datum'])
        
        # Mappa till tydliga namn: 'El kWh' är import, 'Produktion' är export
        df = df.rename(columns={
            'El kWh': 'grid_import_kwh',
            'Produktion': 'grid_export_kwh'
        })

        if 'Totalt (öre/kWh)' in df.columns:
            df = df.rename(columns={'Totalt (öre/kWh)': 'price_total_ore'})
        else:
            print(f"Varning: Hittade inte pris-kolumn i {f}")

        dfs.append(df[['timestamp', 'grid_import_kwh', 'grid_export_kwh', 'price_total_ore']])

    return pd.concat(dfs).drop_duplicates('timestamp')

def load_solis_data(filenames):
    """Läser inverter-data och konverterar W till kW."""
    dfs = []
    for f in filenames:
        print(f"Bearbetar Solis-data: {f}")
        df = pd.read_csv(f)
        df['timestamp'] = pd.to_datetime(df['timestamp'])
        
        # Konvertera W -> kW
        df['total_production_kwh'] = pd.to_numeric(df['avg_power_kw'], errors='coerce') / 1000.0
        dfs.append(df[['timestamp', 'total_production_kwh']])

    return pd.concat(dfs).drop_duplicates('timestamp')

def load_nasa_data(filename):
    """Läser NASA-data och justerar LST (UTC+1) till Europe/Stockholm."""
    print(f"Bearbetar NASA-väderdata: {filename}")
    with open(filename, 'r') as f:
        lines = f.readlines()
    start_idx = next(i for i, line in enumerate(lines) if line.startswith('YEAR'))
    df = pd.read_csv(StringIO(''.join(lines[start_idx:])))
    
    df['timestamp'] = pd.to_datetime(df[['YEAR', 'MO', 'DY', 'HR']].rename(
        columns={'YEAR': 'year', 'MO': 'month', 'DY': 'day', 'HR': 'hour'}))
    
    # Justera tid från fast UTC+1 till svensk tid (hanterar sommar/vintertid)
    df['timestamp'] = df['timestamp'].dt.tz_localize('Etc/GMT-1').dt.tz_convert('Europe/Stockholm').dt.tz_localize(None)
    df = df.rename(columns={'ALLSKY_SFC_SW_DWN': 'irradiance_wm2'})
    return df[['timestamp', 'irradiance_wm2']]

# --- 2. KÖR BEARBETNING ---

# De faktiska filnamnen i din mapp
el_files = [
    'El - Erlandsliden 15 B, Halmstad 2023 - halvår.csv',
    'El - Erlandsliden 15 B, Halmstad 2024.csv',
    'El - Erlandsliden 15 B, Halmstad 2025.csv'
]
solis_files = [
    'solis_data_2023_korrigerad.csv',
    'solis_data_2024_local.csv',
    'solis_data_2025_local.csv'
]
nasa_file = 'POWER_Point_Hourly_20230701_20260323_056d66N_012d81E_LST.csv'

df_grid = load_el_data(el_files)
df_inv = load_solis_data(solis_files)
df_weather = load_nasa_data(nasa_file)

# --- 3. SLÅ SAMMAN OCH BERÄKNA ---

master_df = pd.merge(df_grid, df_inv, on='timestamp', how='left')
master_df = pd.merge(master_df, df_weather, on='timestamp', how='left')

# Fyll natt/hål i produktion med 0
master_df['total_production_kwh'] = master_df['total_production_kwh'].fillna(0)

# A. Egenförbrukning = Produktion - Export
master_df['self_consumption_kwh'] = (master_df['total_production_kwh'] - master_df['grid_export_kwh']).clip(lower=0)

# B. Totala laster = köpt + egenförbrukning
master_df['total_house_load_kwh'] = master_df['grid_import_kwh'] + master_df['self_consumption_kwh']

# Spara
master_df = master_df.sort_values('timestamp').reset_index(drop=True)
master_df.to_csv('master_energy_data_v2.csv', index=False)

print("\n" + "="*40)
print(f"KLART! 'master_energy_data_v2.csv' har skapats.")
print(f"Korrelation Sol/Produktion: {master_df['total_production_kwh'].corr(master_df['irradiance_wm2']):.2f}")
print("="*40)
display(master_df.head())

Bearbetar el-leverantörsdata: El - Erlandsliden 15 B, Halmstad 2023 - halvår.csv
Bearbetar el-leverantörsdata: El - Erlandsliden 15 B, Halmstad 2024.csv
Bearbetar el-leverantörsdata: El - Erlandsliden 15 B, Halmstad 2025.csv
Bearbetar Solis-data: solis_data_2023_korrigerad.csv
Bearbetar Solis-data: solis_data_2024_local.csv
Bearbetar Solis-data: solis_data_2025_local.csv
Bearbetar NASA-väderdata: POWER_Point_Hourly_20230701_20260323_056d66N_012d81E_LST.csv

KLART! 'master_energy_data_v2.csv' har skapats.
Korrelation Sol/Produktion: 0.89


,timestamp,grid_import_kwh,grid_export_kwh,price_total_ore,total_production_kwh,irradiance_wm2,self_consumption_kwh,total_house_load_kwh
0,2023-07-01 00:00:00,0.8,0.0,0.000,0.000000,NaN,0.000000,0.800000
1,2023-07-01 01:00:00,2.4,0.0,34.859,0.000000,0.00,0.000000,2.400000
2,2023-07-01 02:00:00,6.0,0.0,34.481,0.000000,0.00,0.000000,6.000000
3,2023-07-01 03:00:00,2.6,0.0,34.232,0.000000,0.00,0.000000,2.600000
4,2023-07-01 04:00:00,0.6,0.0,34.008,0.023333,1.98,0.023333,0.623333


In [4]:
import pandas as pd
import numpy as np

print("=== UPPDATERA MASTER-FIL MED TEMPERATUR + FULL TVÄTT ===\n")

# --- STEG 0: LÄSA IN NUVARANDE MASTER + NASA-RAW ---
df = pd.read_csv('master_energy_data_v2.csv')
df['timestamp'] = pd.to_datetime(df['timestamp'])

print(f"Master-fil inläst: {len(df)} rader")
print(f"Datumintervall: {df['timestamp'].min()} → {df['timestamp'].max()}\n")

# --- STEG 1: SKAPA TEMPERATUR-PROXY FRÅN MÅNAD + TIMME ---
# Baserat på klassisk norsk/svensk vädermodell (genomsnitt + dagsvariationer)
# Du kan senare byta detta mot riktig NASA T2M eller annat API

def estimate_temperature(month, hour):
    """
    Enkel temperaturmodell baserad på månad och timme.
    Returneraa temperatur i °C.
    """
    # Månadsmedeltemperaturer för Halmstad/Göteborg (ungefärlig)
    monthly_avg = {
        1: 0.5, 2: 0.8, 3: 4.0, 4: 9.0, 5: 15.0, 6: 19.5,
        7: 21.0, 8: 20.0, 9: 15.5, 10: 10.0, 11: 5.5, 12: 1.5
    }
    
    base_temp = monthly_avg[month]
    
    # Dagsvariasjon: kallast ~ 5.00, varmast ~ 14.00
    # Använd sinusfunktion för att modellera denna
    hour_variation = 7 * np.sin(2 * np.pi * (hour - 5) / 24)
    
    return base_temp + hour_variation

# Applicera på master-filen
df['month'] = df['timestamp'].dt.month
df['hour'] = df['timestamp'].dt.hour
df['temp_c'] = df.apply(
    lambda row: estimate_temperature(row['month'], row['hour']),
    axis=1
)

print("Temperatur-proxy skapad från månad + timme")
print(f"Temp-intervall: {df['temp_c'].min():.1f}°C → {df['temp_c'].max():.1f}°C\n")

# --- STEG 2: TVÄTTA IRRADIANCE ---
df['irradiance_wm2'] = df['irradiance_wm2'].replace(-999, np.nan)

missing_before = df['irradiance_wm2'].isnull().sum()
print(f"Saknade irradiance innan: {missing_before}")

df['irradiance_wm2'] = df['irradiance_wm2'].interpolate(method='linear')

# Nollställ mörka timmar
night_mask = (df['total_production_kwh'] == 0) & (df['hour'].isin([22, 23, 0, 1, 2, 3, 4, 5]))
df.loc[night_mask, 'irradiance_wm2'] = 0.0

df['irradiance_wm2'] = df['irradiance_wm2'].fillna(0)
df['irradiance_wm2'] = df['irradiance_wm2'].clip(lower=0)

missing_after = df['irradiance_wm2'].isnull().sum()
print(f"Saknade irradiance efter:  {missing_after}\n")

# --- STEG 3: KLIPP NEGATIVA ENERGIVÄRDEN ---
energy_cols = [
    'grid_import_kwh', 'grid_export_kwh', 'total_production_kwh',
    'self_consumption_kwh', 'total_house_load_kwh'
]
df[energy_cols] = df[energy_cols].clip(lower=0)

# --- STEG 4: PRICE-TVÄTT ---
df['price_total_ore'] = df['price_total_ore'].fillna(0)

# --- STEG 5: FEATURE ENGINEERING ---
df['weekday'] = df['timestamp'].dt.weekday
df['is_weekend'] = (df['weekday'] >= 5).astype(int)
df['season'] = df['month'].map({
    12: 'winter', 1: 'winter', 2: 'winter',
    3: 'spring', 4: 'spring', 5: 'spring',
    6: 'summer', 7: 'summer', 8: 'summer',
    9: 'autumn', 10: 'autumn', 11: 'autumn'
})

# Cyklisk kodning av timme
df['hour_sin'] = np.sin(2 * np.pi * df['hour'] / 24)
df['hour_cos'] = np.cos(2 * np.pi * df['hour'] / 24)

# Nettobalans
df['net_balance_kwh'] = df['total_production_kwh'] - df['total_house_load_kwh']

# --- STEG 6: SORTERA OCH SPARA ---
df = df.sort_values('timestamp').reset_index(drop=True)

# Välj kolumner i logisk ordning
cols_order = [
    'timestamp', 'hour', 'weekday', 'is_weekend', 'month', 'season',
    'hour_sin', 'hour_cos', 'temp_c', 'irradiance_wm2',
    'grid_import_kwh', 'grid_export_kwh', 'total_production_kwh',
    'self_consumption_kwh', 'total_house_load_kwh', 'net_balance_kwh',
    'price_total_ore'
]
df = df[cols_order]

df.to_csv('master_energy_data_cleaned.csv', index=False)

print("=== TVÄTT + UPPDATERING KLAR ===")
print(f"Sparad: master_energy_data_cleaned.csv")
print(f"Rader: {len(df)}")
print(f"Kolumner: {len(df.columns)}\n")

print("Kolumnöversikt:")
print(df[['timestamp', 'temp_c', 'irradiance_wm2', 'total_production_kwh',
          'grid_import_kwh', 'net_balance_kwh', 'season']].head(10))

print("\n--- STATISTIK ---")
print(df.describe().round(2))

print("\n*** VIKTIGT ***")
print("temp_c är en PROXY (årstidsmodell + dagsvariasjon).")
print("Ersätt senare med riktig temperaturdata från NASA T2M eller SMHI.")
print("Net_balance > 0 = överskott (batteri laddar), < 0 = underskott (batteri eller nätet)")

=== UPPDATERA MASTER-FIL MED TEMPERATUR + FULL TVÄTT ===

Master-fil inläst: 21961 rader
Datumintervall: 2023-07-01 00:00:00 → 2025-12-31 23:00:00

Temperatur-proxy skapad från månad + timme
Temp-intervall: -6.5°C → 28.0°C

Saknade irradiance innan: 25
Saknade irradiance efter:  0

=== TVÄTT + UPPDATERING KLAR ===
Sparad: master_energy_data_cleaned.csv
Rader: 21961
Kolumner: 17

Kolumnöversikt:
            timestamp     temp_c  irradiance_wm2  total_production_kwh  \
0 2023-07-01 00:00:00  14.238519            0.00              0.000000   
1 2023-07-01 01:00:00  14.937822            0.00              0.000000   
2 2023-07-01 02:00:00  16.050253            0.00              0.000000   
3 2023-07-01 03:00:00  17.500000            0.00              0.000000   
4 2023-07-01 04:00:00  19.188267            1.98              0.023333   
5 2023-07-01 05:00:00  21.000000           18.42              0.176667   
6 2023-07-01 06:00:00  22.811733           62.67              0.239167   
7 2023-07-